In [4]:
import numpy as np
import pandas as pd
from pathlib import Path

def eisenstein_Ep_square(Z: np.ndarray, p: int, M: int = 12) -> np.ndarray:
    """
    Ep(z) = sum_{(m,n)!=(0,0)} 1 / (z + m + i n)^p
    Obcięcie: |m|,|n| <= M
    """
    Z = np.asarray(Z, dtype=np.complex128)
    Ep = np.zeros_like(Z, dtype=np.complex128)

    shifts = range(-M, M + 1)

    with np.errstate(invalid="ignore", divide="ignore", over="ignore"):
        for m in shifts:
            for n in shifts:
                if m == 0 and n == 0:
                    continue
                Ep += 1.0 / (Z + (m + 1j * n))**p

    return Ep

In [5]:
def compute_epp_mod_from_csv(csv_path: Path, p: int, M: int = 12) -> dict:
    df = pd.read_csv(csv_path)

    a = df["a_re"].to_numpy() + 1j * df["a_im"].to_numpy()
    r = df["r_norm"].to_numpy()
    N = len(a)
    if N < 2:
        raise ValueError("Za mało obiektów (N < 2).")

    # wagi
    w = np.pi * r**2
    f = w.sum()

    # macierz różnic
    Z = a[:, None] - a[None, :]
    np.fill_diagonal(Z, np.nan + 1j*np.nan)

    Ep = eisenstein_Ep_square(Z, p=p, M=M)
    Ep = np.where(np.isfinite(Ep.real), Ep, 0.0 + 0.0j)

    # wyzeruj przekątną (konsekwentnie)
    np.fill_diagonal(Ep, 0.0 + 0.0j)

    # u_j = sum_i w_i Ep(i,j)  => (w^T Ep) jako wektor po kolumnach
    u = (w[:, None] * Ep).sum(axis=0)                 # (N,)

    # v_j = sum_k w_k conj(Ep(j,k))
    v = (w[None, :] * np.conjugate(Ep)).sum(axis=1)   # (N,)

    epp_mod = ( (w**(p-1)) * u * v ).sum() / (f**(p+1))

    return {
        "N": int(N),
        "M": int(M),
        "p": int(p),
        "f": float(f),
        "epp_mod_re": float(epp_mod.real),
        "epp_mod_im": float(epp_mod.imag),
        "epp_mod_abs": float(np.abs(epp_mod)),
    }


In [6]:
res33 = compute_epp_mod_from_csv(args.csv, p=3, M=args.M)
res88 = compute_epp_mod_from_csv(args.csv, p=8, M=args.M)
print("\n=== epp_mod (p=3) ===", res33)
print("\n=== epp_mod (p=8) ===", res88)

NameError: name 'args' is not defined